# Spatial Metabolomics: WikiPathways Enrichment Analysis

This notebook performs metabolite set enrichment analysis on Day60 vs Day1 differential metabolites identified in the companion notebook (`spatial_metabolomics_clean_v2.ipynb`).

**Approach:**
- Background: all compounds detectable in the instrument m/z window (100–1500 Da) from the RaMP-DB database (v2.5.4)
- Pathway database: WikiPathways (filtered from RaMP-DB)
- Test: one-sided Fisher's exact test per pathway, with Benjamini–Hochberg FDR correction
- Minimum pathway size: 5 detectable metabolites (pathways below this are skipped)

**Cell types analyzed:**
| Label | Description |
|-------|-------------|
| `epithelial` | Epithelial cells |
| `adipocyte` | Adipocytes |
| `blood_vessel` | Vascular/endothelial regions |
| `stromal` | Stromal and other non-classified cells |

**Outputs:**
- Per-cell-type enrichment bar plots (FDR < 0.05)
- Pathway redundancy analysis (Jaccard overlap between top pathways)
- LFC heatmap: hit molecules × top enriched pathways, colored by fold change
- Composite lipid figure integrating MALDI-MSI and bulk LC-MS lipidomics

## Imports

In [ ]:
import sqlite3
import re

import numpy as np
import pandas as pd
from scipy.stats import fisher_exact
from scipy.cluster.hierarchy import linkage, leaves_list
from scipy.spatial.distance import pdist
from statsmodels.stats.multitest import multipletests

import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.gridspec as gridspec
from matplotlib.patches import Patch
import seaborn as sns

mpl.rcParams['pdf.fonttype'] = 42

## Configuration

In [ ]:
# ── Input paths ───────────────────────────────────────────────────────────────
RAMP_DB_PATH   = '../data/RaMP_SQLite_v2.5.4.sqlite'
DE_POS_CSV     = '../results/DE_day60vsday1_percelltype_pos.csv'
DE_NEG_CSV     = '../results/DE_day60vsday1_percelltype_neg.csv'
LCMS_CSV       = '../data/lipidomics_lfc.csv'   # bulk LC-MS lipidomics log2FC table

# ── Output paths ───────────────────────────────────────────────────────────────
RESULTS_DIR    = '../results/'
FIGS_DIR       = './'

# ── Analysis parameters ───────────────────────────────────────────────────────
MZ_MIN         = 100    # instrument detectable window (Da)
MZ_MAX         = 1500
MIN_PATHWAY_SIZE = 5    # minimum detectable metabolites per pathway
FDR_THRESHOLD  = 0.05

CELLTYPES      = ['epithelial', 'adipocyte', 'blood_vessel', 'stromal']

---
## 1. Helper Functions

In [ ]:
def run_pathway_enrichment(formula_db, de_formulas, bg_formulas):
    """
    One-sided Fisher's exact test for metabolite set enrichment.

    Parameters
    ----------
    formula_db : pd.DataFrame
        Pathway database with columns ['pathwayName', 'pathwayId', 'mol_formula'].
    de_formulas : set
        Differentially expressed molecular formulas (query set).
    bg_formulas : set
        Background/universe set (all metabolites detectable in the experiment).

    Returns
    -------
    pd.DataFrame
        Per-pathway results sorted by FDR, with columns:
        pathwayName, pathwayId, num_hits, pathway_size_detected,
        fold_enrichment, p_value, fdr, hit_formulas.
    """
    results = []
    valid_universe = set(bg_formulas)
    total_de_in_bg = len(de_formulas.intersection(valid_universe))
    total_bg_size  = len(valid_universe)
    global_ratio   = total_de_in_bg / total_bg_size if total_bg_size > 0 else 0

    for (name, pid), group in formula_db.groupby(['pathwayName', 'pathwayId']):
        pathway_formulas           = set(group['mol_formula'].unique())
        detectable_pathway_formulas = pathway_formulas.intersection(valid_universe)

        if len(detectable_pathway_formulas) < MIN_PATHWAY_SIZE:
            continue

        hits     = de_formulas.intersection(detectable_pathway_formulas)
        num_hits = len(hits)
        if num_hits == 0:
            continue

        a = num_hits
        b = total_de_in_bg - a
        c = len(detectable_pathway_formulas) - a
        d = total_bg_size - (a + b + c)

        _, p_val = fisher_exact([[a, b], [c, d]], alternative='greater')

        pathway_ratio  = num_hits / len(detectable_pathway_formulas)
        fold_enrichment = pathway_ratio / global_ratio if global_ratio > 0 else 0

        results.append({
            'pathwayName':            name,
            'pathwayId':              pid,
            'num_hits':               a,
            'pathway_size_detected':  len(detectable_pathway_formulas),
            'fold_enrichment':        round(fold_enrichment, 2),
            'p_value':                p_val,
            'hit_formulas':           sorted(hits),
        })

    res_df = pd.DataFrame(results)
    if not res_df.empty:
        _, res_df['fdr'], _, _ = multipletests(res_df['p_value'], method='fdr_bh')
        res_df = res_df.sort_values(['fdr', 'fold_enrichment'], ascending=[True, False])

    return res_df


def get_pathway_source(pid):
    """Infer pathway database from pathway ID prefix."""
    if pid.startswith('WP'):    return 'WikiPathways'
    if pid.startswith('R-HSA'): return 'Reactome'
    if pid.startswith('SMP'):   return 'SMPDB'
    if pid.startswith('map'):   return 'KEGG'
    return 'Other'

In [ ]:
def plot_pathway_enrichment(res_df, celltype, top_n=15, figsize=(7, 5)):
    """
    Horizontal bar plot of top enriched pathways for one cell type.

    Parameters
    ----------
    res_df : pd.DataFrame
        Output of run_pathway_enrichment().
    celltype : str
        Cell type label for plot title.
    top_n : int
        Maximum number of pathways to display.
    figsize : tuple
        Figure dimensions in inches.
    """
    df = (res_df
          .query("fdr < @FDR_THRESHOLD")
          .nsmallest(top_n, "fdr")
          .copy()
          .sort_values("fdr", ascending=False))

    if df.empty:
        print(f"No significant pathways (FDR < {FDR_THRESHOLD}) for {celltype}")
        return None

    df["neg_log10_fdr"] = -np.log10(df["fdr"])
    df["label"]         = df["pathwayName"].str.slice(0, 55)

    norm   = mcolors.Normalize(vmin=df["fold_enrichment"].min(),
                                vmax=df["fold_enrichment"].max())
    cmap   = plt.cm.viridis
    colors = [cmap(norm(fe)) for fe in df["fold_enrichment"]]

    fig, ax = plt.subplots(figsize=figsize)
    bars = ax.barh(df["label"], df["neg_log10_fdr"],
                   color=colors, edgecolor="white", linewidth=0.5, height=0.65)

    for bar, (_, row) in zip(bars, df.iterrows()):
        ax.text(bar.get_width() + 0.05, bar.get_y() + bar.get_height() / 2,
                f"{int(row['num_hits'])}/{int(row['pathway_size_detected'])}",
                va="center", ha="left", fontsize=8, color="#5F5E5A")

    sm   = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
    sm.set_array([])
    cbar = fig.colorbar(sm, ax=ax, shrink=0.6, pad=0.02)
    cbar.set_label("Fold enrichment", fontsize=9)
    cbar.ax.tick_params(labelsize=8)

    ax.set_xlabel("−log₁₀(FDR)", fontsize=11)
    ax.set_title(f"Pathway enrichment — {celltype}\n"
                 f"(top {len(df)} significant pathways, WikiPathways)",
                 fontsize=11, fontweight="bold", pad=10)
    ax.tick_params(axis="y", labelsize=9)
    ax.tick_params(axis="x", labelsize=9)
    ax.spines[["top", "right"]].set_visible(False)
    ax.set_xlim(0, df["neg_log10_fdr"].max() * 1.25)
    plt.tight_layout()
    return fig

---
## 2. Load Pathway Database (RaMP-DB)

We use [RaMP-DB v2.5.4](https://rampdb.nih.gov/) as the source of metabolite–pathway associations and compound properties. The database is queried via its SQLite distribution.

**Two queries are made:**
1. Compound–pathway mapping (ChEBI compounds with molecular formulas, filtered to WikiPathways)
2. All detectable compounds in the instrument m/z window (used as background universe)

In [ ]:
con = sqlite3.connect(RAMP_DB_PATH)

# ── 1. Compound–pathway mapping ───────────────────────────────────────────────
query_map = """
SELECT
    s.sourceId          AS chebi_id,
    p.pathwayName       AS pathwayName,
    p.sourceId          AS pathwayId,
    p.type              AS pathwayType,
    c.mol_formula,
    c.monoisotop_mass,
    c.common_name
FROM source s
JOIN analytehaspathway ahp  ON s.rampId = ahp.rampId
JOIN pathway p              ON ahp.pathwayRampId = p.pathwayRampId
JOIN chem_props c           ON s.sourceId = c.chem_source_id
WHERE s.geneOrCompound = 'compound'
AND   s.sourceId LIKE 'chebi:%'
AND   c.mol_formula IS NOT NULL
"""
full_map = pd.read_sql_query(query_map, con)

# ── 2. All compounds in detectable m/z window ─────────────────────────────────
query_bg = """
SELECT DISTINCT
    s.sourceId      AS chebi_id,
    c.mol_formula,
    c.monoisotop_mass
FROM source s
JOIN chem_props c  ON s.sourceId = c.chem_source_id
WHERE s.geneOrCompound = 'compound'
AND   c.mol_formula IS NOT NULL
AND   c.monoisotop_mass IS NOT NULL
"""
all_compounds = pd.read_sql_query(query_bg, con)
con.close()

print(f"Pathway–formula pairs: {len(full_map):,}")
print(f"Unique pathways:       {full_map['pathwayId'].nunique():,}")
print(f"All compounds in DB:   {len(all_compounds):,}")

In [ ]:
# Filter pathway map to WikiPathways only
full_map['pathway_source'] = full_map['pathwayId'].apply(get_pathway_source)
full_map_sub = (full_map[['pathwayName', 'pathwayId', 'mol_formula']]
                .drop_duplicates()
                .dropna(subset=['mol_formula'])
                .query('pathwayId.str.startswith("WP")', engine='python'))

print(f"WikiPathways pathway–formula pairs: {len(full_map_sub):,}")
print(f"Unique WikiPathways:                {full_map_sub['pathwayId'].nunique():,}")

In [ ]:
# Build background universe: all compounds detectable by the instrument
theoretical_bg = set(
    all_compounds.query('@MZ_MIN <= monoisotop_mass <= @MZ_MAX')['mol_formula']
)
print(f"Background universe size: {len(theoretical_bg):,} unique molecular formulas")

In [ ]:
# Build formula → representative name lookup (used for plot labels)
def pick_first_name(names):
    if isinstance(names, list): return names[0] if names else None
    if isinstance(names, str):  return names
    return None

full_map['rep_name'] = full_map['common_name'].apply(pick_first_name)
formula_to_name = (
    full_map
    .dropna(subset=['mol_formula', 'rep_name'])
    .drop_duplicates(subset=['mol_formula'])
    .set_index('mol_formula')['rep_name']
    .to_dict()
)
print(f"Formula-to-name entries: {len(formula_to_name):,}")

---
## 3. Load Differential Expression Results

Day60 vs Day1 Wilcoxon DE results from both ion panels (output of Notebook 1, Section 9).

In [ ]:
df_de_pos = pd.read_csv(DE_POS_CSV)
df_de_neg = pd.read_csv(DE_NEG_CSV)

# Background: all metabolites observed in either panel
bg_formulas = set(df_de_pos.names) | set(df_de_neg.names)

print(f"Positive panel metabolites: {df_de_pos.names.nunique():,}")
print(f"Negative panel metabolites: {df_de_neg.names.nunique():,}")
print(f"Combined background:        {len(bg_formulas):,} unique formulas")
df_de_pos.head()

---
## 4. Per-cell-type Pathway Enrichment

For each cell type, DE metabolites from both ion panels (FDR < 0.05, any direction) are pooled and tested against the WikiPathways background.

In [ ]:
# Run enrichment for all cell types and store results
res_dict = {}
for ct in CELLTYPES:
    de_formulas = (
        set(df_de_pos.query("celltype == @ct & pvals_adj <= @FDR_THRESHOLD").names) |
        set(df_de_neg.query("celltype == @ct & pvals_adj <= @FDR_THRESHOLD").names)
    )
    print(f"{ct}: {len(de_formulas)} DE formulas (pos+neg union)")
    res_dict[ct] = run_pathway_enrichment(full_map_sub, de_formulas, theoretical_bg)
    n_sig = (res_dict[ct]['fdr'] < FDR_THRESHOLD).sum() if not res_dict[ct].empty else 0
    print(f"  → {len(res_dict[ct])} pathways tested, {n_sig} significant at FDR < {FDR_THRESHOLD}\n")

In [ ]:
# Save combined enrichment results
res_all = pd.concat(
    [df.assign(region=ct) for ct, df in res_dict.items() if not df.empty],
    ignore_index=True
)
res_all.to_csv(RESULTS_DIR + 'pathway_enrichment_day60vsday1_all_celltypes.csv', index=False)
print(f"Saved {len(res_all)} pathway entries across all cell types")

### 4a. Per-cell-type bar plots

In [ ]:
for ct in CELLTYPES:
    fig = plot_pathway_enrichment(res_dict[ct], celltype=ct, top_n=10)
    if fig is not None:
        fig.savefig(FIGS_DIR + f'pathway_enrichment_{ct}.pdf', dpi=300, bbox_inches='tight')
        plt.show()

---
## 5. Pathway Redundancy Analysis

Top enriched pathways often share hit molecules (e.g., multiple lipid pathways driven by the same TAG species). This section checks pairwise molecule overlap between the top N pathways per cell type using the Jaccard index, and visualizes pathway × molecule membership matrices.

A mean Jaccard > 0.8 across top pathways indicates they are largely driven by the same molecules and should be interpreted with caution.

In [ ]:
def check_pathway_redundancy(res_df, celltype, top_n=5, formula_to_name=None):
    """
    Assess molecule overlap between the top N enriched pathways.

    Outputs:
    - Summary table of hit molecules per pathway
    - Pairwise Jaccard index matrix
    - Molecule × pathway membership heatmap
    """
    top = res_df.nsmallest(top_n, 'fdr').copy()
    if top.empty:
        print(f"No results for {celltype}")
        return None, None

    def fmt(f):
        if formula_to_name is None: return f
        name = formula_to_name.get(f)
        return f"{name} ({f})" if name else f

    # Summary table
    print(f"\n{'='*60}")
    print(f"Region: {celltype} — top {top_n} pathways")
    print(f"{'='*60}")
    for _, row in top.iterrows():
        hits = sorted(row['hit_formulas'])
        print(f"\n  {row['pathwayName']}")
        print(f"    FDR={row['fdr']:.4f}  hits={row['num_hits']}/{row['pathway_size_detected']}")
        print(f"    Molecules: {[fmt(f) for f in hits]}")

    all_hit_sets = [set(row['hit_formulas']) for _, row in top.iterrows()]
    union_hits   = set.union(*all_hit_sets)
    shared_hits  = set.intersection(*all_hit_sets)

    print(f"\n── Molecule overlap ──────────────────────────────────────────")
    print(f"  Unique formulas across top {top_n}: {len(union_hits)}")
    print(f"  Shared by ALL {top_n}:              {len(shared_hits)}")
    if shared_hits:
        print(f"  Shared: {[fmt(f) for f in sorted(shared_hits)]}")

    # Pairwise Jaccard
    names = [row['pathwayName'][:35] for _, row in top.iterrows()]
    n = len(top)
    J = np.zeros((n, n))
    for i in range(n):
        for j in range(n):
            inter = len(all_hit_sets[i] & all_hit_sets[j])
            union = len(all_hit_sets[i] | all_hit_sets[j])
            J[i, j] = inter / union if union > 0 else 0

    off_diag = J[np.triu_indices(n, k=1)]
    mean_J   = off_diag.mean()
    print(f"\n  Mean pairwise Jaccard: {mean_J:.2f}")
    if mean_J > 0.8:
        print("  ⚠ Top pathways are largely driven by the same molecules.")
    elif mean_J > 0.5:
        print("  ⚠ Substantial overlap — pathways are not fully independent.")
    else:
        print("  ✓ Pathways appear driven by reasonably distinct molecule sets.")

    # Membership heatmap + Jaccard heatmap
    formula_list  = sorted(union_hits)
    pathway_names = [row['pathwayName'][:45] for _, row in top.iterrows()]

    membership = pd.DataFrame(
        [[1 if f in hits else 0 for hits in all_hit_sets] for f in formula_list],
        index=formula_list, columns=pathway_names
    )
    membership = membership.loc[membership.sum(axis=1).sort_values(ascending=False).index]
    idx_labels = [fmt(f) for f in membership.index]

    fig, axes = plt.subplots(1, 2,
                             figsize=(15, max(4, len(formula_list) * 0.2 + 1.5)),
                             gridspec_kw={'width_ratios': [4, 2]})

    ax = axes[0]
    for col_i, pathway in enumerate(pathway_names):
        for row_i, formula in enumerate(membership.index):
            val   = membership.loc[formula, pathway]
            color = '#1C3A5F' if val else '#F1EFE8'
            ax.add_patch(plt.Rectangle((col_i, row_i), 1, 1,
                                        facecolor=color, edgecolor='white', linewidth=1.2))
    for row_i, formula in enumerate(membership.index):
        if membership.loc[formula].sum() == top_n:
            ax.axhspan(row_i, row_i+1, color='#FFF3CD', zorder=0, alpha=0.5)
    ax.set_xlim(0, len(pathway_names)); ax.set_ylim(0, len(membership))
    ax.set_xticks(np.arange(len(pathway_names)) + 0.5)
    ax.set_xticklabels(pathway_names, rotation=30, ha='right', fontsize=8.5)
    ax.set_yticks(np.arange(len(membership)) + 0.5)
    ax.set_yticklabels(idx_labels, fontsize=8)
    ax.spines[:].set_visible(False)
    ax.set_title(f'{celltype} — molecule × pathway membership\n'
                 f'(yellow = shared across all {top_n} pathways)',
                 fontsize=10, fontweight='bold')

    ax2 = axes[1]
    im  = ax2.imshow(J, cmap='RdYlBu_r', vmin=0, vmax=1, aspect='auto')
    short = [n[:20] for n in names]
    ax2.set_xticks(range(n)); ax2.set_xticklabels(short, rotation=30, ha='right', fontsize=8)
    ax2.set_yticks(range(n)); ax2.set_yticklabels(short, fontsize=8)
    for i in range(n):
        for j in range(n):
            ax2.text(j, i, f'{J[i,j]:.2f}', ha='center', va='center', fontsize=8,
                     color='white' if J[i,j] > 0.6 else '#1C3A5F')
    plt.colorbar(im, ax=ax2, shrink=0.6).set_label('Jaccard', fontsize=8)
    ax2.set_title('Pairwise formula\noverlap (Jaccard)', fontsize=10, fontweight='bold')
    plt.suptitle(f'Are top {top_n} pathways driven by the same molecules? — {celltype}',
                 fontsize=11, fontweight='bold', y=1.01)
    plt.tight_layout()
    fig.savefig(FIGS_DIR + f'pathway_redundancy_{celltype}.pdf', dpi=300, bbox_inches='tight')
    plt.show()

    return membership, pd.DataFrame(J, index=names, columns=names)

In [ ]:
for ct, res_df in res_dict.items():
    check_pathway_redundancy(res_df, ct, top_n=20, formula_to_name=formula_to_name)

---
## 6. Pathway LFC Heatmap

For the top N enriched pathways per cell type, we build a molecule × pathway matrix where cell values are log fold changes (Day60 vs Day1) from the Wilcoxon DE results.

If a molecular formula is detected in both positive and negative ion panels, both rows are shown with mode indicated by row label color (brown = positive ion mode, blue = negative ion mode).

In [ ]:
# Curated short lipid names for display (formula → canonical lipid name)
formula_to_shortname = {
    "C55H104O6": "TAG(16:0/18:0/18:1)",
    "C57H100O6": "TAG(18:1/18:2/18:2)",
    "C53H98O6":  "TAG(16:0/16:1/18:1)",
    "C55H102O6": "TAG(18:1/18:1/16:0)",
    "C57H102O6": "TAG(18:1/18:1/18:2)",
    "C57H106O6": "TAG(18:1/18:1/18:0)",
    "C57H98O6":  "TAG(18:2/18:2/18:2)",
    "C51H96O6":  "TAG(16:0/16:0/16:1)",
    "C10H15N5O10P2": "PAP (adenosine 3',5'-bisphosphate)",
    "C21H27N7O14P2": "NAD",
    "C39H73O8P":  "PA(18:0/18:2)",
    "C43H78NO7P": "PE(P-18:1/20:4)",
    "C20H32O2":   "FA(20:4) — arachidonic acid",
    "C18H32O2":   "FA(18:2) — linoleic acid",
    "C20H34O2":   "FA(20:3)",
    "C22H34O2":   "FA(22:5) — DPA n-3",
    "C22H32O2":   "FA(22:6) — DHA",
    "C18H34O2":   "FA(18:1) — oleic acid",
    "C46H78NO10P": "PS(18:0/22:6)",
    "C44H84NO8P":  "PC(18:1/18:1)",
    "C44H86NO8P":  "PC(18:1/18:0)",
    "C44H82NO8P":  "PC(16:0/20:3)",
    "C44H80NO8P":  "PC(20:4/16:0)",
    "C42H82NO8P":  "PC(16:1/18:0)",
    "C44H78NO8P":  "PC(16:0/20:5)",
    "C42H80NO8P":  "PC(16:1/18:1)",
    "C41H78NO8P":  "PC(15:0/18:2)",
    "C40H80NO8P":  "PC(16:0/16:0)",
    "C48H86NO8P":  "PC(18:1/22:4)",
}

In [ ]:
def make_pathway_lfc_table(res_df, celltype, top_n, df_de_pos, df_de_neg,
                          formula_to_name=None):
    """
    Build a molecule × pathway LFC table for the top N enriched pathways.

    If a formula is detected in both ion panels, two rows are produced
    (labeled [pos] and [neg]). Gray cells indicate non-membership in that pathway.

    Parameters
    ----------
    res_df : pd.DataFrame
        Enrichment results for one cell type.
    celltype : str
        Cell type label (must match 'celltype' column in DE tables).
    top_n : int
        Number of top pathways to include (ranked by FDR).
    df_de_pos, df_de_neg : pd.DataFrame
        Wilcoxon DE results for positive and negative panels.
    formula_to_name : dict, optional
        Formula → display name mapping for row labels.

    Returns
    -------
    pd.DataFrame
        LFC matrix (rows = molecules, columns = pathways).
    """
    top = res_df.nsmallest(top_n, 'fdr').copy()
    if top.empty:
        print(f"No results for {celltype}")
        return None

    pathway_names = list(top['pathwayName'])
    all_hit_sets  = [set(row['hit_formulas']) for _, row in top.iterrows()]
    union_hits    = sorted(set.union(*all_hit_sets))

    pos_lfc = (df_de_pos.query("celltype == @celltype")
               .drop_duplicates('names').set_index('names')['logfoldchanges'].to_dict())
    neg_lfc = (df_de_neg.query("celltype == @celltype")
               .drop_duplicates('names').set_index('names')['logfoldchanges'].to_dict())

    def fmt(f):
        if formula_to_name:
            name = formula_to_name.get(f)
            return f"{name} ({f})" if name else f
        return f

    rows, row_labels, row_modes = [], [], []
    for f in union_hits:
        in_pos = f in pos_lfc
        in_neg = f in neg_lfc
        membership_vec = [1 if f in hits else 0 for hits in all_hit_sets]

        if in_pos and in_neg:
            for mode, lfc_dict in [('pos', pos_lfc), ('neg', neg_lfc)]:
                rows.append([lfc_dict[f] if m else np.nan for m in membership_vec])
                row_labels.append(f"{fmt(f)} [{mode}]")
                row_modes.append(mode)
        elif in_pos:
            rows.append([pos_lfc[f] if m else np.nan for m in membership_vec])
            row_labels.append(fmt(f)); row_modes.append('pos')
        elif in_neg:
            rows.append([neg_lfc[f] if m else np.nan for m in membership_vec])
            row_labels.append(fmt(f)); row_modes.append('neg')

    lfc_df = pd.DataFrame(rows, index=row_labels, columns=pathway_names)
    lfc_df['_n'] = (~lfc_df.isna()).sum(axis=1)
    lfc_df['_m'] = lfc_df.drop(columns=['_n']).abs().mean(axis=1)
    lfc_df = lfc_df.sort_values(['_n', '_m'], ascending=[False, False]).drop(columns=['_n', '_m'])
    row_modes = [row_modes[list(lfc_df.index).index(l)] for l in lfc_df.index]

    # Plot
    n_rows, n_cols = len(lfc_df), len(pathway_names)
    abs_max = max(lfc_df.stack().abs().max(), 0.1)
    norm    = plt.Normalize(vmin=-abs_max, vmax=abs_max)
    cmap    = plt.cm.RdBu_r
    mode_colors = {'pos': '#854F0B', 'neg': '#185FA5'}

    fig, ax = plt.subplots(figsize=(max(n_cols + 3, 10), max(n_rows * 0.28 + 1.5, 5)))
    for col_i, pathway in enumerate(pathway_names):
        for row_i, label in enumerate(lfc_df.index):
            val = lfc_df.loc[label, pathway]
            if pd.isna(val):
                ax.add_patch(plt.Rectangle((col_i, row_i), 1, 1,
                                            facecolor='#F1EFE8', edgecolor='white', linewidth=1.2))
            else:
                ax.add_patch(plt.Rectangle((col_i, row_i), 1, 1,
                                            facecolor=cmap(norm(val)), edgecolor='white', linewidth=1.2))
                text_color = 'white' if abs(val) > abs_max * 0.5 else '#1C3A5F'
                ax.text(col_i + 0.5, row_i + 0.5, f'{val:+.2f}',
                        ha='center', va='center', fontsize=7.5,
                        color=text_color, fontweight='bold')

    ax.set_xlim(0, n_cols); ax.set_ylim(0, n_rows)
    ax.set_xticks(np.arange(n_cols) + 0.5)
    ax.set_xticklabels([p[:40] for p in pathway_names], rotation=35, ha='right', fontsize=9)
    ax.set_yticks(np.arange(n_rows) + 0.5)
    ax.set_yticklabels(lfc_df.index, fontsize=8)
    for tick, mode in zip(ax.get_yticklabels(), row_modes):
        tick.set_color(mode_colors.get(mode, '#444441'))
    ax.spines[:].set_visible(False)

    sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm); sm.set_array([])
    cbar = fig.colorbar(sm, ax=ax, shrink=0.5, pad=0.02)
    cbar.set_label('log fold change (Day60 vs Day1)', fontsize=9)
    cbar.ax.tick_params(labelsize=8)

    legend_elements = [
        Patch(facecolor='white', edgecolor='white', label='Row label color:'),
        Patch(facecolor='white', edgecolor='white', label='  brown = positive ion mode'),
        Patch(facecolor='white', edgecolor='white', label='  blue  = negative ion mode'),
        Patch(facecolor='#F1EFE8', edgecolor='#D3D1C7', label='Gray = not a hit in this pathway'),
    ]
    ax.legend(handles=legend_elements, loc='upper right',
              bbox_to_anchor=(1.0, -0.22), fontsize=8, frameon=False, ncol=2)
    ax.set_title(f'{celltype} — LFC of hit molecules per pathway\n'
                 f'(top {top_n} pathways · red = up · blue = down · gray = not a hit)',
                 fontsize=10, fontweight='bold', pad=10)
    plt.tight_layout()
    fig.savefig(FIGS_DIR + f'pathway_lfc_{celltype}.pdf', dpi=300, bbox_inches='tight')
    plt.show()
    return lfc_df

In [ ]:
for ct in CELLTYPES:
    lfc_table = make_pathway_lfc_table(
        res_df          = res_dict[ct],
        celltype        = ct,
        top_n           = 5,
        df_de_pos       = df_de_pos,
        df_de_neg       = df_de_neg,
        formula_to_name = formula_to_shortname,
    )
    if lfc_table is not None:
        lfc_table.to_csv(FIGS_DIR + f'pathway_lfc_{ct}.csv')

---
## 7. Composite Lipid Figure: MALDI-MSI + Bulk LC-MS Integration

We integrate MALDI-MSI regional LFC data with bulk LC-MS lipidomics (log2FC, Day60 vs Day1) for the same lipid species. This allows cross-validation between the spatial and bulk platforms.

A canonical lipid name normalization step reconciles the naming conventions between the two datasets before matching by molecular formula.

In [ ]:
def normalize_lipid_name(name):
    """
    Convert various lipid naming conventions to a canonical form for formula matching.

    Examples:
        'PC 14:0-14:0'  → 'PC(14:0/14:0)'
        'LPC 22:4'      → 'LPC(22:4)'
        'PC(16:0/20:3)' → 'PC(16:0/20:3)'  (already canonical)
    """
    name = name.strip()
    if re.match(r'^[A-Z]+\(', name):
        return name
    m = re.match(r'^([A-Za-z]+)\s+(.+)$', name)
    if m:
        lipid_class = m.group(1).upper()
        chains      = re.sub(r'-(?=\d)', '/', m.group(2))
        return f"{lipid_class}({chains})"
    return name


def build_lcms_name_map(lcms_df, formula_to_shortname):
    """Match LC-MS lipid names to molecular formulas via canonical name normalization."""
    canonical_to_formula = {
        normalize_lipid_name(v): k for k, v in formula_to_shortname.items()
    }
    df = lcms_df.copy()
    df['Lipid_canonical'] = df['Lipid'].apply(normalize_lipid_name)
    df['formula']         = df['Lipid_canonical'].map(canonical_to_formula)

    matched   = df['formula'].notna().sum()
    unmatched = df['formula'].isna().sum()
    print(f"Matched:   {matched} / {len(df)} LC-MS lipids")
    print(f"Unmatched: {unmatched}")
    if unmatched > 0:
        print("\nUnmatched entries (canonical form shown):")
        for _, row in df[df['formula'].isna()].iterrows():
            print(f"  '{row['Lipid']}' → '{row['Lipid_canonical']}'")
    return df

In [ ]:
lcms_df        = pd.read_csv(LCMS_CSV, index_col=0)
lcms_df_mapped = build_lcms_name_map(lcms_df, formula_to_shortname)
lcms_df_mapped.head()

In [ ]:
def plot_composite_lipid_figure(
    res_df, celltype_list, df_de_pos, df_de_neg,
    lcms_formula_lookup, top_n=12, formula_to_name=None,
    figsize=(24, 14), output_prefix='composite_lipid'
):
    """
    Three-panel composite figure:
    - Top: LC-MS log2FC bar chart (bar color = −log10 p-value)
    - Middle: MALDI-MSI pathway membership matrix (black = member)
    - Bottom: MALDI-MSI regional LFC heatmap (red = up, blue = down)

    Columns are molecular formulas, clustered by correlation distance of MALDI LFC profiles.
    """
    top_pathways  = res_df.nsmallest(top_n, 'fdr')
    pathway_names = list(top_pathways['pathwayName'])
    all_hit_sets  = [set(row['hit_formulas']) for _, row in top_pathways.iterrows()]
    union_hits    = sorted(set.union(*all_hit_sets))

    def fmt(f):
        return formula_to_name.get(f, f) if formula_to_name else f

    # MALDI LFC matrix (rows = cell types, columns = formulas)
    lfc_matrix = pd.DataFrame(index=celltype_list, columns=union_hits, dtype=float)
    for ct in celltype_list:
        pos = (df_de_pos.query("celltype == @ct").drop_duplicates('names')
               .set_index('names')['logfoldchanges'])
        neg = (df_de_neg.query("celltype == @ct").drop_duplicates('names')
               .set_index('names')['logfoldchanges'])
        for f in union_hits:
            vals = ([pos[f]] if f in pos.index else []) + ([neg[f]] if f in neg.index else [])
            if vals: lfc_matrix.loc[ct, f] = np.mean(vals)

    # Cluster columns by correlation distance
    valid_cols   = lfc_matrix.columns[lfc_matrix.notna().any(axis=0)]
    lfc_valid    = lfc_matrix[valid_cols].fillna(0)
    col_order    = leaves_list(linkage(pdist(lfc_valid.T.values, metric='correlation'), method='average'))
    ordered_formulas = [valid_cols[i] for i in col_order] + [f for f in union_hits if f not in valid_cols]

    n_cols, n_pathways, n_regions = len(ordered_formulas), len(pathway_names), len(celltype_list)
    membership   = pd.DataFrame({f: [1 if f in hits else 0 for hits in all_hit_sets]
                                  for f in ordered_formulas}, index=pathway_names)
    lfc_ordered  = lfc_matrix[ordered_formulas]

    lcms_fc = np.array([lcms_formula_lookup.loc[f, 'log2FC']
                        if f in lcms_formula_lookup.index else np.nan
                        for f in ordered_formulas], dtype=float)
    lcms_pv = np.array([lcms_formula_lookup.loc[f, 'p_value']
                        if f in lcms_formula_lookup.index else np.nan
                        for f in ordered_formulas], dtype=float)
    neg_log10_p = -np.log10(np.where(lcms_pv > 0, lcms_pv, np.nan))

    fig    = plt.figure(figsize=figsize, facecolor='white')
    outer  = gridspec.GridSpec(1, 2, width_ratios=[n_cols, 1.2], wspace=0.02, figure=fig)
    inner  = gridspec.GridSpecFromSubplotSpec(3, 1, subplot_spec=outer[0],
                                              height_ratios=[1.8, n_pathways * 0.45, n_regions * 0.9],
                                              hspace=0.03)
    cbar_gs = gridspec.GridSpecFromSubplotSpec(2, 1, subplot_spec=outer[1], hspace=0.6)

    ax_bar = fig.add_subplot(inner[0])
    ax_mem = fig.add_subplot(inner[1])
    ax_lfc = fig.add_subplot(inner[2])
    ax_cb1 = fig.add_subplot(cbar_gs[0])
    ax_cb2 = fig.add_subplot(cbar_gs[1])
    ax_mem.sharex(ax_bar); ax_lfc.sharex(ax_bar)

    # ── Panel A: LC-MS bars ───────────────────────────────────────────────────
    bar_cmap = plt.cm.YlOrRd_r
    pv_max   = np.nanmax(neg_log10_p) if np.any(~np.isnan(neg_log10_p)) else 1
    pv_norm  = plt.Normalize(vmin=0, vmax=pv_max)
    for i, (fc, pv_s) in enumerate(zip(lcms_fc, neg_log10_p)):
        if np.isnan(fc): continue
        color = bar_cmap(pv_norm(pv_s)) if not np.isnan(pv_s) else '#CCCCCC'
        ax_bar.bar(i + 0.5, fc, width=0.7, color=color, linewidth=0, zorder=2)
    ax_bar.axhline(0, color='#888780', linewidth=0.8, zorder=1)
    ax_bar.set_xlim(0, n_cols)
    ax_bar.set_ylabel('LC-MS\nlog₂FC', fontsize=8)
    ax_bar.spines[['top', 'right', 'bottom']].set_visible(False)
    ax_bar.tick_params(axis='y', labelsize=7)
    ax_bar.tick_params(axis='x', labelbottom=False, bottom=False)
    ax_bar.set_xticks([])
    ax_bar.set_title('LC-MS lipidomics log₂FC  ·  bar color = −log₁₀(p-value)',
                     fontsize=8, pad=4, loc='left', color='#444441')
    sm_bar = plt.cm.ScalarMappable(cmap=bar_cmap, norm=pv_norm); sm_bar.set_array([])
    cbar1 = fig.colorbar(sm_bar, cax=ax_cb1)
    cbar1.set_label('−log₁₀(p)\nLC-MS', fontsize=7); cbar1.ax.tick_params(labelsize=6)

    # ── Panel B: pathway membership ───────────────────────────────────────────
    for col_i, f in enumerate(ordered_formulas):
        for row_i, pathway in enumerate(pathway_names):
            color = '#1a1816' if membership.loc[pathway, f] else '#F5F3EF'
            ax_mem.add_patch(plt.Rectangle((col_i, row_i), 1, 1,
                                            facecolor=color, edgecolor='white', linewidth=0.6))
    ax_mem.set_ylim(0, n_pathways)
    ax_mem.set_yticks(np.arange(n_pathways) + 0.5)
    ax_mem.set_yticklabels(pathway_names, fontsize=7.5)
    ax_mem.spines[:].set_visible(False)
    ax_mem.set_ylabel('Pathway\nmembership', fontsize=8)
    ax_mem.tick_params(axis='x', bottom=False, labelbottom=False)

    # ── Panel C: MALDI LFC heatmap ────────────────────────────────────────────
    abs_max  = max(np.nanmax(np.abs(lfc_ordered.values)), 0.5)
    lfc_norm = plt.Normalize(vmin=-abs_max, vmax=abs_max)
    lfc_cmap = plt.cm.RdBu_r
    for col_i, f in enumerate(ordered_formulas):
        for row_i, ct in enumerate(celltype_list):
            val   = lfc_ordered.loc[ct, f]
            color = '#D3D1C7' if pd.isna(val) else lfc_cmap(lfc_norm(val))
            ax_lfc.add_patch(plt.Rectangle((col_i, row_i), 1, 1,
                                            facecolor=color, edgecolor='white', linewidth=0.6))
    ax_lfc.set_ylim(0, n_regions)
    ax_lfc.set_yticks(np.arange(n_regions) + 0.5)
    ax_lfc.set_yticklabels([ct.replace('_', ' ').capitalize() for ct in celltype_list], fontsize=8)
    ax_lfc.spines[:].set_visible(False)
    ax_lfc.set_ylabel('MALDI-MSI\nregion', fontsize=8)
    ax_lfc.set_xticks(np.arange(n_cols) + 0.5)
    ax_lfc.set_xticklabels([fmt(f) for f in ordered_formulas], rotation=90, fontsize=6.5, ha='center')
    sm_lfc = plt.cm.ScalarMappable(cmap=lfc_cmap, norm=lfc_norm); sm_lfc.set_array([])
    cbar2 = fig.colorbar(sm_lfc, cax=ax_cb2)
    cbar2.set_label('Mean log FC\nMALDI-MSI', fontsize=7); cbar2.ax.tick_params(labelsize=6)

    legend_elements = [
        Patch(facecolor='#1a1816', label='Pathway member'),
        Patch(facecolor='#F5F3EF', edgecolor='#CCCCCC', label='Not a member'),
        Patch(facecolor='#D3D1C7', label='Not detected (MALDI-MSI)'),
    ]
    ax_lfc.legend(handles=legend_elements, bbox_to_anchor=(1.0, -0.55),
                  loc='lower right', fontsize=7, frameon=False)
    fig.suptitle(f'Lipid species — pathway membership and regional abundance changes\n'
                 f'(top {top_n} pathways · molecules clustered by correlation distance)',
                 fontsize=10, fontweight='bold', y=1.01)
    plt.savefig(f'{output_prefix}.pdf', dpi=300, bbox_inches='tight', facecolor='white')
    plt.savefig(f'{output_prefix}.png', dpi=300, bbox_inches='tight', facecolor='white')
    plt.show()
    print(f"Saved {output_prefix}.pdf / .png")
    return membership, lfc_ordered

In [ ]:
# Build union of top 5 pathways per cell type for the composite figure
top_per_celltype = pd.concat([
    res_dict[ct].assign(celltype=ct).sort_values('fdr').head(5)
    for ct in CELLTYPES if not res_dict[ct].empty
])
top_pathways_union = (top_per_celltype
    .sort_values('fdr')
    .drop_duplicates('pathwayName')
    .sort_values('fdr'))

# Prepare LC-MS lookup indexed by molecular formula
lcms_formula_lookup = (lcms_df_mapped
    .dropna(subset=['formula'])
    .drop_duplicates('formula')
    .set_index('formula')[['log2FC', 'p_value']])

membership, lfc_table = plot_composite_lipid_figure(
    res_df              = top_pathways_union,
    celltype_list       = ['adipocyte', 'stromal', 'epithelial', 'blood_vessel'],
    df_de_pos           = df_de_pos,
    df_de_neg           = df_de_neg,
    lcms_formula_lookup = lcms_formula_lookup,
    top_n               = 20,
    formula_to_name     = formula_to_shortname,
    figsize             = (24, 14),
    output_prefix       = FIGS_DIR + 'composite_lipid_figure',
)

---
## 8. Cross-cell-type LFC Correlation

As a supplementary visualization, we examine how correlated the Day60 vs Day1 LFC profiles are across cell types — using the union of significant DE metabolites from both panels.

In [ ]:
# Pool significant DE metabolites from both panels
df_combined = pd.concat([df_de_pos, df_de_neg])
df_combined = df_combined.groupby(['names', 'celltype']).mean(numeric_only=True).reset_index()

sig_names   = list(set(
    df_de_pos.query('pvals_adj <= @FDR_THRESHOLD').names
) | set(
    df_de_neg.query('pvals_adj <= @FDR_THRESHOLD').names
))
print(f"Significant metabolites (union, pos+neg): {len(sig_names)}")

lfc_matrix_all = df_combined.pivot(index='names', columns='celltype', values='logfoldchanges')
lfc_matrix_sig = lfc_matrix_all.loc[sig_names]
lfc_matrix_sig.head()

In [ ]:
# Pearson correlation between cell type LFC profiles (significant metabolites)
dfcorr = lfc_matrix_sig.corr(method='pearson')

fig = plt.figure(figsize=(4, 4))
sns.clustermap(dfcorr, cmap='RdBu_r', square=True,
               linewidths=0.1, linecolor='white',
               center=0, figsize=(4, 4))
plt.suptitle('Cell type LFC correlation\n(Day60 vs Day1, significant metabolites)',
             y=1.02, fontsize=10)
plt.savefig(FIGS_DIR + 'celltype_lfc_correlation.pdf', dpi=300, bbox_inches='tight')
plt.show()